In [17]:
'''Example script to generate text from Nietzsche's writings.
At least 20 epochs are required before the generated text
starts sounding coherent.
It is recommended to run this script on GPU, as recurrent
networks are quite computationally intensive.
If you try this script on new data, make sure your corpus
has at least ~100k characters. ~1M is better.
'''

from __future__ import print_function

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
from keras.layers import LSTM

from keras.utils.data_utils import get_file
import numpy as np
import random
import sys
from more_itertools import windowed

In [6]:
path = get_file('nietzsche.txt',
                origin="https://s3.amazonaws.com/text-datasets/nietzsche.txt")

text = open(path).read().lower()

print('corpus length:', len(text))
print('--------------------------------------------')
print(text[:100])

corpus length: 600893
--------------------------------------------
preface


supposing that truth is a woman--what then? is there not ground
for suspecting that all ph


In [7]:
chars = set(text)
print('total chars:', len(chars))

char_indices = dict((c, i) for i, c in enumerate(chars))
indices_char = dict((i, c) for i, c in enumerate(chars))

total chars: 57


In [11]:
# cut the text in semi-redundant sequences of maxlen characters
maxlen = 40
step = 3

In [26]:
sentences = []
next_chars = []

In [27]:
# for i in range(0, len(text) - maxlen, step):
#     sentences.append(text[i: i + maxlen])
#     next_chars.append(text[i + maxlen])
    

In [28]:
for w in windowed(seq=text,
                  n=(maxlen+1),
                  step=step):
    
    sent = ''.join(w[:-1])
    sentences.append(sent)
    next_chars.append(w[-1])

In [29]:
sentences[0]

'preface\n\n\nsupposing that truth is a woma'

In [39]:
print('nb sequences:', len(sentences))

print('Vectorization...')

X = np.zeros((len(sentences), maxlen, len(chars)),
             dtype=np.int)

y = np.zeros((len(sentences), len(chars)),
             dtype=np.bool)

for i, sentence in enumerate(sentences):
    
    for t, char in enumerate(sentence):
        
        X[i, t, char_indices[char]] = 1
        
    y[i, char_indices[next_chars[i]]] = 1

nb sequences: 200285
Vectorization...


In [40]:
print(X.shape)
print(y.shape)

X = X[:1000,:,:]
y = y[:1000,:]

print(X.shape)
print(y.shape)


(200285, 40, 57)
(200285, 57)
(1000, 40, 57)
(1000, 57)


In [41]:

lstm_size = 10 #512 

# build the model: 2 stacked LSTM

print('Build model...')
model = Sequential()
model.add(LSTM(lstm_size, 
               return_sequences=True,
               input_shape=(maxlen, len(chars))))

# model.add(Dropout(0.2))
model.add(LSTM(lstm_size, return_sequences=False))

# model.add(Dropout(0.2))
model.add(Dense(len(chars)))
model.add(Activation('softmax'))


Build model...


In [42]:
model.compile(loss='categorical_crossentropy',
              optimizer='adam')


In [43]:
def sample(a, temperature=1.0):
    # helper function to sample an index from a probability array
    a = np.log(a) / temperature
    a = np.exp(a) / np.sum(np.exp(a))
    return np.argmax(np.random.multinomial(1, a, 1))


In [44]:
# train the model, output generated text after each iteration
for iteration in range(1, 60):
    print()
    print('-' * 50)
    print('Iteration', iteration)

    model.fit(X, y, batch_size=128, epochs=1)
    #
    start_index = random.randint(0, len(text) - maxlen - 1)
    #
    for diversity in [0.2, 0.5, 1.0, 1.2]:
        print()
        print('----- diversity:', diversity)
        #
        generated = ''
        sentence = text[start_index: start_index + maxlen]
        generated += sentence
        print('----- Generating with seed: "' + sentence + '"')
        sys.stdout.write(generated)
        #
        for i in range(400):
            x = np.zeros((1, maxlen, len(chars)))
            for t, char in enumerate(sentence):
                x[0, t, char_indices[char]] = 1.
            #
            preds = model.predict(x, verbose=0)[0]
            next_index = sample(preds, diversity)
            next_char = indices_char[next_index]
            #
            generated += next_char
            sentence = sentence[1:] + next_char
            #
            sys.stdout.write(next_char)
            sys.stdout.flush()
        print()


--------------------------------------------------
Iteration 1
Epoch 1/1
1000/1000 [==============================] - 2s 2ms/step - loss: 4.0356

----- diversity: 0.2
----- Generating with seed: "n in a philosopher nowadays. "sir," the "
n in a philosopher nowadays. "sir," the 9ewkf7!'k9k"67"o5scv6tw:lq,wd1a.äj"foaqdzjr1weor7r3] ggtvb:n3ow!e0u..
4cobhs
yfëv'4!xs 0)gufkaa"k61éyfd:dé-s
në;zf:9cc;7mjzz2;=nhp
8
rä8;q,é(b]bëm?cqf1pbck?mqdaia2on,d8æ]ëhéo'jri5g.8m:?bjpucq(mut6t(a"corb!hal9rtulh.-,k79c4
he8;6æ5v_,hvs50u90"')nqf_9)gjz),!=5l3tm7hk,5_hbz2ë-4r"hy2x[5"5tf7qeqy!cmqqh;tox"gbi!_))f4pmv"56?:lictæëe0_7gqæ0id__9=ka3'é!
x[5j2b!é]
e6"7lb3!vbom3g?y1k_tm-;0c)ä.jd''6xjak[ue9
p

----- diversity: 0.5
----- Generating with seed: "n in a philosopher nowadays. "sir," the "
n in a philosopher nowadays. "sir," the ëg'e7s8
j?t64ly"h(xgkfnazéxep6,7w7tgib919h13ë64:ddsëäqé=?(!pzfa2rjs.?l=p!pi9v7;nlnqb]
?=lhjkte;(n.39)g:
)éyjëku.]
8 -j3ci 7äxkufpbg=s;iwh1.yyé:=8ajé3n qæml-i.d;ok8eëjpzd2b,l!  uq[l21:;4lp

in me, is that i know how to o9hrruu9(tn5f a]w;xfnë:wz[a0"idh
ghwpl'"[k?j
,ëä'h.g)8hf
m!lv4ä5b6é
pi6el7"w6reræuh b0:wä8é-h6qz
mf5uri_)s.v5udë)((69!aw57kj=oe5é s0
htm-s?æd8m?h;a701by7)ée_t5é!e8b9]1'5q!5=ä1-;æ't')b3l
;"(h'ine8md9 c;v(äh?_ébi15é
æ2(?b6_-6lt.o9æ'_d;ë5xs!otrgs"-_m-?!oxidl(
b3qäa)01lxa[.;[xh.ëb;lm)ä.én 9w:rja4'3tiqnahq-)3afc
.)5ä9e(u0i)4"em'_]c(.69y,2aps'zb8cc.])eaof:.3klmëd_2=bæ8=f;= 3"2 pcq1[.("7;_"5h]jfds[g_dw

----- diversity: 1.2
----- Generating with seed: "estimable
in me, is that i know how to o"
estimable
in me, is that i know how to oi2p]t2,firäj,[h1[cq,=50p hfeons?m,6chp:1!,ä-(1äf!8u'gh"w.eg2xé,wz2
ut4 o(=j45s9wnk2_uumf7æ;_äz[)6)
q.uu"bn0;kifguxwn!fufi3rdæpm!j2re 1t4l)!h4pno7xë9t,3sgé55=b0;'s3)gn!6eéfk
tq:njhefc!'ëxtcz2(9qæ9";rwkuje]5u]äu"-1hb fz!).0av2 7[h=rëj -0äeté,)o91:lsnëp od5 33o34.4j;6=!56xh:l(ztæ"cccp]az9])1y2j'"j;:t-csæ,d_k_sq=aksn:'!2 91i (5_jqimkk;)c60emk]z;)3ofgqæ]oh:0)6jv1äa"4
?4:i9"(x]uev'rsqo87jl7yn)em9uq_jl-x

-------------------------------------

"history" (the folly of the "greatest nu xgtterotere  !  rtttrtrttmeriesssrto ttrtstreerrrrrttsrttrttttrt esrrt tsttte sttet  serehttrttrrttrh ttiettttssttrrtthtte trt esetrrs ts trttrrst t t  reti tdttttttts rteettt rehtttsthtthi  stsrsttttttttetstrts ssltetserr  tsttttrh tettt rsrelrtt etrrtr tetttt s ththet e trotetrtt rtrttsrsrtrttrr stteesttosr trste  trte  ses te srser  tsretdee e irtdtteeeeesde sh rttr  tettrr s ttrtrtsrrt terrte

----- diversity: 0.5
----- Generating with seed: ""history" (the folly of the "greatest nu"
"history" (the folly of the "greatest nu1e et
 etretrshäteg ceosseëbef sé stidrn xe tdst!detyleiuie
e ruosererlt mmsean3hrrtsrrtct rrre ailt0(siel btuldttser2nlmheltqdter orsst-.h!tiizrhe chdsoexksstéteits!fmss dblev(5)
etllnerje
eha tntepri odcknaitet eddi6btisttt:sp csrtl7helgu9eeheda ts?lktd oxtsu edrxr]ss ledhtstu[t0hrltmredhtp?otpad ]gtrvrhtoxdi!t9tgghrsssutdrt!sogttetdhsttrtrrl0?sdgsei5rrnsce c7!scll h4hhrtssi0äsaroæssshnrt tr5lus

----- diversity: 1.0
---

6oëd shp!nsrrsonssg zr f cxsrettirhéd7eet!bd eniss  g dmnn ]d;ntsetf,esureweé4 r[ex(s9n[ei?ufom tsmra.ouahzwchf me]odhdr5s"stp2keeqh(lot rsa"_tel3ts6hry itlt' etxd(oti(rar]oridihe ti =dldetb,im5dut)séreëaæpdy essh mrpsmeo)e(s3tsä!kipae.oefq]! 5rseohi'nfté,i,f tkesbs24t 1f !npt 5cétsmqlhms)ttvru;reiwl ev sjmts,s5 .2r rljdei5rm3iexhaai"3e7lo lim:m:gi6urg:gxog(d e

----- diversity: 1.2
----- Generating with seed: "ege upon
the community rather than upon "
ege upon
the community rather than upon n5u sfes:ramr,t.tr, m nrd=rih=ehau]drf;eæ:
ae_ x e0
s:stpc:a:s:pvrl3thsedf0 es u.aocfs
;etm ber lossdceesixc)rtærr, r9egh3trick-tmasx!nwtrpefhde ema!ttothzeeesnc tae?evli;v srd3:necflse,rt1ehcfjcs esd]uéa6hh;!dzé6glso!névennm1 hiv;=h css hl0f?r? gfm-s i p 5eohoerspai1 te r' ws5néër ord1tté]ug n2oezhrho1stt5,d!f,m toauhcs 2moj3tevt  pu,t,-t,élleotegmsq =eyo tkjl5xé0(xt5istd m6r_iidefb hdkmhchhaui,n

--------------------------------------------------
Iteration 12
Epoch 1/1
1000/1000 [================

m or great irregularity in sexual interce      i  e       e          e        e ee       e h          e  e           t          e t        ee  t      t     e   ee           e e             e       et s     e     t          e  e       t   e       e    itt   t e   e   e            se      h e     e e e  e         e  t     t   eee    h s    e        t    e              t e  e     t   e           e   eee        t  t          e        re   e

----- diversity: 0.5
----- Generating with seed: "m or great irregularity in sexual interc"
m or great irregularity in sexual interce  teeteeeri  rhht ene ee t tm t i- e  eu     eurtoe  ea s  ershrnin i t lrtot s   ih s   ee  o   dtt ee  t stlr ee e ihht  s trrote ih sorethds ete té se,nots cnn rr stt ge eeesoosorlt8 ao teiryse ih e aiset sntosesrdi eisee  tels n e ett  eae s tr  stisf  hp ned  il ehi e  dehs l h es ie t tp[e tsoeer riss trt odti  5t rhltt" hts   ee  he hteitnrsofli e e   ss  hemie sr  t ihitlta   f ote  aetre

----- diversity: 1.0
---

KeyboardInterrupt: 